MCP SCENARIO: “Smart HR Onboarding Assistant”
🧩 Scenario Background
You are working in a company called XYZ Corp.
New employees often face challenges during onboarding, such as:
- Trouble accessing payroll portal
- Confusion about leave policies
- Difficulty setting up email accounts
- Questions about training schedules
👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

🤖 What this Bot Should Do
When a new hire types a question/problem:
- Understand the query (e.g., “I can’t log into payroll”)
- Decide if escalation to HR is needed
- Identify:
- Category (Payroll / Policy / IT Setup / Training)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, step-by-step instructions)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

In [2]:
!pip install -q groq

import os
import json
from datetime import datetime
from groq import Groq
from google.colab import userdata

# ============================================
# STEP 0: CLIENT SETUP
# ============================================

api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# ============================================
# STEP 1: DATABASE (Simulated storage)
# ============================================

tickets_db = []

# ============================================
# STEP 2: TOOL LAYER
# ============================================

def create_support_ticket(issue, priority, category, context):
    """
    MCP Tool:
    Simulates HR onboarding support ticket creation.
    In real-world, this could connect to HRMS / ServiceNow / internal HR portal.
    """
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category,
        "created_at": datetime.now().isoformat(),
        "source": "hr_onboarding_bot",
        "context": context
    }

    tickets_db.append(ticket)
    return ticket

# ============================================
# STEP 3: CONTEXT OBJECT
# ============================================

def build_context(user_input):
    """
    MCP-style context object.
    """
    return {
        "user_input": user_input,
        "session_id": f"session_{len(tickets_db) + 1}",
        "timestamp": datetime.now().isoformat(),
        "agent_name": "HROnboardingAgent"
    }

# ============================================
# STEP 4: GUIDANCE LAYER
# ============================================

def provide_guidance(category):
    faq_map = {
        "payroll": (
            "Try these steps:\n"
            "1. Open the payroll portal from your welcome email\n"
            "2. Use your employee ID as username\n"
            "3. Click 'Forgot Password' if login fails\n"
            "4. If access is still blocked, HR/payroll support will help"
        ),
        "policy": (
            "Try these steps:\n"
            "1. Open the employee handbook or policy portal\n"
            "2. Check leave policy under HR policies section\n"
            "3. Review casual, sick, and earned leave details\n"
            "4. If still unclear, raise an HR support request"
        ),
        "it setup": (
            "Try these steps:\n"
            "1. Check your welcome email for account credentials\n"
            "2. Use the temporary password for first login\n"
            "3. Set up mail in Outlook or company portal\n"
            "4. If login/setup still fails, IT setup support is needed"
        ),
        "training": (
            "Try these steps:\n"
            "1. Open the onboarding calendar shared by HR\n"
            "2. Check your email for training invites\n"
            "3. Confirm your reporting manager's schedule\n"
            "4. If training details are missing, HR can assist"
        ),
        "general": (
            "Please share a bit more detail about your onboarding issue so I can classify it correctly."
        )
    }
    return faq_map.get(category, faq_map["general"])

# ============================================
# STEP 5: LLM ANALYSIS LAYER
# ============================================

def analyze_with_llm(user_input, context):
    """
    LLM decides:
    - create_ticket
    - category
    - priority
    - short_reason
    """

    prompt = f"""
You are an enterprise HR onboarding assistant.

Analyze the new employee's onboarding issue and return ONLY valid JSON.

Rules:
- create_ticket = true if the issue clearly needs escalation to HR or IT support
- create_ticket = false if the issue is only informational or can be solved with basic guidance
- category must be one of:
  "payroll", "policy", "it setup", "training", "general"
- priority must be one of:
  "high", "medium"

Return exactly this JSON schema:
{{
  "create_ticket": true,
  "category": "payroll",
  "priority": "high",
  "short_reason": "User cannot access payroll portal and onboarding is blocked"
}}

Context:
{json.dumps(context, indent=2)}

User Input:
"{user_input}"
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Return only valid JSON. No markdown. No explanation."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    output = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(output)
    except Exception:
        try:
            start = output.find("{")
            end = output.rfind("}") + 1
            parsed = json.loads(output[start:end])
        except Exception:
            parsed = {
                "create_ticket": True,
                "category": "general",
                "priority": "medium",
                "short_reason": "Fallback used due to JSON parse issue"
            }

    parsed["category"] = str(parsed.get("category", "general")).lower().strip()
    parsed["priority"] = str(parsed.get("priority", "medium")).lower().strip()

    if parsed["category"] not in {"payroll", "policy", "it setup", "training", "general"}:
        parsed["category"] = "general"

    if parsed["priority"] not in {"high", "medium"}:
        parsed["priority"] = "medium"

    parsed["create_ticket"] = bool(parsed.get("create_ticket", True))
    parsed["short_reason"] = parsed.get("short_reason", "No reason provided")

    return parsed

# ============================================
# STEP 6: EXECUTION METADATA
# ============================================

def build_metadata(context, decision):
    return {
        "agent_name": context["agent_name"],
        "session_id": context["session_id"],
        "timestamp": datetime.now().isoformat(),
        "decision_summary": decision["short_reason"]
    }

# ============================================
# STEP 7: MCP ORCHESTRATOR
# ============================================

def mcp_hr_agent(user_input):
    """
    Main MCP flow:
    User -> Context -> LLM Analysis -> Decision -> Tool Call -> Final Response
    """

    context = build_context(user_input)
    print("\n🧠 Agent received:", user_input)
    print("🗂️ Context:", context)

    decision = analyze_with_llm(user_input, context)
    print("🤖 LLM Decision:", decision)

    guidance = provide_guidance(decision["category"])

    metadata = build_metadata(context, decision)
    print("📝 Metadata:", metadata)

    if decision["create_ticket"]:
        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"],
            "context": context
        }

        print("📦 MCP Payload:", payload)

        result = create_support_ticket(**payload)

        return f"""
✅ Support Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
Created At: {result['created_at']}

Why ticket was created:
- {decision['short_reason']}

Instant Guidance:
{guidance}

Next Step:
- HR/IT onboarding team will review this case shortly.
- Keep your ticket ID for follow-up.
"""

    else:
        return f"""
🤖 No Ticket Required Right Now

Reason:
- {decision['short_reason']}

Suggested Guidance:
{guidance}

Next Step:
- Try the above onboarding guidance first.
- If the problem continues, raise the issue again with more detail.
"""

# ============================================
# STEP 8: RUN LOOP
# ============================================

print("🚀 LLM + MCP-style HR Onboarding Assistant Started (type 'exit')\n")

while True:
    user_input = input("Enter onboarding issue: ").strip()

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    if not user_input:
        print("⚠️ Please enter a valid issue.")
        continue

    response = mcp_hr_agent(user_input)
    print(response)

🚀 LLM + MCP-style HR Onboarding Assistant Started (type 'exit')

Enter onboarding issue: Can you explain the leave policy for new employees?

🧠 Agent received: Can you explain the leave policy for new employees?
🗂️ Context: {'user_input': 'Can you explain the leave policy for new employees?', 'session_id': 'session_1', 'timestamp': '2026-03-28T06:34:46.929724', 'agent_name': 'HROnboardingAgent'}
🤖 LLM Decision: {'create_ticket': False, 'category': 'policy', 'priority': 'medium', 'short_reason': 'User requests information on leave policy for new employees'}
📝 Metadata: {'agent_name': 'HROnboardingAgent', 'session_id': 'session_1', 'timestamp': '2026-03-28T06:34:47.271447', 'decision_summary': 'User requests information on leave policy for new employees'}

🤖 No Ticket Required Right Now

Reason:
- User requests information on leave policy for new employees

Suggested Guidance:
Try these steps:
1. Open the employee handbook or policy portal
2. Check leave policy under HR policies section


KeyboardInterrupt: Interrupted by user